# Tracking et Model Registry

Le pipeline Kedro `training` envoie métriques, paramètres et modèle vers MLflow. Le pipeline `deployment` enregistre ce run dans le Model Registry et le passe en staging ou en production selon `ENV`.
Prérequis cloud : VM MLflow, bucket d'artifacts, clé du compte de service `purchase-predict` dans `conf/local/service-account.json`.
Ajouter à ce compte le rôle Créateur des objets de l'espace de stockage, puis retélécharger la clé. Ne pas la committer.


## Tracking dans le pipeline training

`python-dotenv` charge `.env` au démarrage du package. Les URL de serveurs restent dans l'environnement ; `parameters.yml` garde les paramètres de pipeline.


Fichier `.env` à la racine du projet.


Fichier `src/purchase_predict/pipelines/training/__init__.py`.


Ajout dans `parameters.yml`.


Signature de `auto_ml` dans `nodes.py`. Sans les arguments MLflow, aucun envoi n'est fait.


Début de `auto_ml`, après la construction de `X` et `y`.


Fin de `auto_ml`. La fonction renvoie aussi `mlflow_run_id`.


Fonction à ajouter dans `nodes.py`. Les courbes partent dans `data/08_reporting`.


Fichier `pipeline.py` du pipeline `training`.


La variable peut aussi figurer dans `.env`. Pour un essai, fixer temporairement `automl_max_evals` à 1, lancer le pipeline `global`, puis rétablir la valeur.


## Registre de modèles

Le Model Registry versionne un modèle et lui associe un état : staging, production ou archived.
Créer le modèle `purchase_predict` depuis l'artifact `model` du dernier run, ou laisser le pipeline `deployment` le faire.
`push_to_model_registry` exige que le run soit déjà dans le tracking. `stage_model` lit `ENV` (`staging` ou `production`).


Fichier `src/purchase_predict/pipelines/deployment/__init__.py`.


Fichier `src/purchase_predict/pipelines/deployment/nodes.py`.


Fichier `.env` complété.


Ajout dans `parameters.yml`.


Fichier `src/purchase_predict/pipelines/deployment/pipeline.py`.


Entrée du catalogue : le run d'entraînement reste en mémoire pour le déploiement.


Enregistrement dans `hooks.py`.


Lancer le pipeline `global` (`automl_max_evals` à 1 le temps de l'essai). Avec `ENV=staging`, la version enregistrée passe en Staging.


## Dépôt

Le registre sépare l'entraînement Kedro du chargement du modèle : un autre projet peut récupérer la version staging ou production.
Pousser le code, puis créer la branche `staging`. Ajouter la clé SSH à l'agent avant le push.
